# Silver Layer - Invoice Table Cleaning

## Source & Destination
- **Source**: `automobile_repair.bronze.invoice`
- **Destination**: `automobile_repair.silver.invoice`

## 9 Essential Columns for Revenue KPIs

### Identifiers (2)
1. `invoice_id` - Primary key
2. `order_id` - Foreign key to orders

### Revenue Data (2)
3. `invoice_date` - Date (converted from TIMESTAMP)
4. `invoice_amount` - Revenue amount

### Time Dimensions (2)
5. `invoice_year` - Year for filtering
6. `invoice_month` - Month for filtering

### Store & Manager Context (3)
7. `store_id` - Store where order was serviced
8. `manager_id` - Manager responsible
9. `manager_name` - Manager name

## Data Quality Issues Fixed
✅ **Converted TIMESTAMP to DATE**
✅ **Deduplicated by invoice_id**
✅ **Added time dimensions** (year, month)
✅ **Linked to store & manager** (for revenue by store/manager KPIs)

## Note on Multiple Invoices
- 50% of orders have 2 invoices (partial payments)
- Each invoice row is kept separately
- **Sum invoice_amount by order_id** in your KPI queries to get total revenue per order

In [0]:
%sql
-- Silver Invoice Transformation
-- For Revenue KPIs: MTD Performance, YTD Growth, Revenue vs Budget

CREATE SCHEMA IF NOT EXISTS automobile_repair.silver;

CREATE OR REPLACE TABLE automobile_repair.silver.invoice AS
WITH deduplicated_invoices AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY invoice_id ORDER BY invoice_date DESC) as rn
    FROM automobile_repair.bronze.invoice
),
base_invoices AS (
    SELECT
        invoice_id,
        order_id,
        CAST(invoice_date AS DATE) AS invoice_date,
        invoice_amount
    FROM deduplicated_invoices
    WHERE rn = 1
      AND invoice_amount IS NOT NULL  -- Exclude NULL amounts (though none exist)
)
SELECT 
    -- Identifiers (2)
    i.invoice_id,
    i.order_id,
    
    -- Revenue data (2)
    i.invoice_date,
    i.invoice_amount,
    
    -- Time dimensions (2)
    YEAR(i.invoice_date) AS invoice_year,
    MONTH(i.invoice_date) AS invoice_month,
    
    -- Store & Manager context (3)
    o.store_id,
    s.manager_id,
    s.manager_name
    
FROM base_invoices i
LEFT JOIN automobile_repair.bronze.order o ON i.order_id = o.order_id
LEFT JOIN automobile_repair.bronze.store s ON o.store_id = s.store_id
ORDER BY i.invoice_date DESC, i.invoice_id;

In [0]:
%sql
-- Verify the 9-column table structure
-- Show recent invoices with store & manager info
SELECT *
FROM automobile_repair.silver.invoice
ORDER BY invoice_date DESC
LIMIT 20;

In [0]:
%sql
-- Monthly revenue by store and manager
-- Ready for MTD Performance and Revenue vs Budget KPIs
SELECT 
    invoice_year,
    invoice_month,
    store_id,
    manager_name,
    COUNT(DISTINCT order_id) as total_orders,
    COUNT(*) as total_invoices,
    SUM(invoice_amount) as total_revenue,
    ROUND(AVG(invoice_amount), 2) as avg_invoice_amount
FROM automobile_repair.silver.invoice
GROUP BY invoice_year, invoice_month, store_id, manager_name
ORDER BY invoice_year DESC, invoice_month DESC, total_revenue DESC
LIMIT 20;